# 얼굴 등록 (InsightFace 버전)

**PC 웹캠**으로 선생님 얼굴을 촬영해 `face_db/known/<이름>/` 에 저장합니다.  
등록 후 `ros2 run pinky_yolo ai_node` 실행 시 추종 대상을 선택할 수 있습니다.

In [ ]:
import sys
import cv2
import numpy as np
from pathlib import Path
from IPython.display import Image as IPyImage, display, clear_output

try:
    from insightface.app import FaceAnalysis
except ImportError:
    print("❌ insightface 미설치 → pip install insightface onnxruntime")
    sys.exit(1)

FACE_DB_DIR = Path.home() / "dev_ws/wasab/src/roscamp-repo-3/Service/WasabAIServer/FaceDB"
KNOWN_DIR   = FACE_DB_DIR / "known"
CACHE_PATH  = FACE_DB_DIR / "encodings.pkl"

KNOWN_DIR.mkdir(parents=True, exist_ok=True)
print("✅ 환경설정 완료")
print(f"📁 저장 위치: {KNOWN_DIR}")

In [ ]:
# InsightFace 모델 로드 (buffalo_sc: 경량 모델)
print("InsightFace 모델 로드 중...")
app = FaceAnalysis(name="buffalo_sc", providers=["CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))
print("✅ 모델 로드 완료")

In [ ]:
# 카메라 연결 확인
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

ok, frame = cap.read()
cap.release()

if ok and frame is not None:
    _, buf = cv2.imencode(".jpg", frame)
    display(IPyImage(data=buf.tobytes()))
    print(f"✅ 카메라 정상 | 해상도: {frame.shape[1]}x{frame.shape[0]}")
else:
    print("❌ 카메라 연결 실패 — 카메라 인덱스를 확인하세요 (VideoCapture(0) → (1) 등)")

In [ ]:
# 등록할 이름 입력
teacher_name = input("등록할 선생님 이름을 입력하세요 (예: kim_teacher): ").strip()

if not teacher_name:
    raise ValueError("❌ 이름이 비어 있습니다.")

user_dir = KNOWN_DIR / teacher_name
user_dir.mkdir(parents=True, exist_ok=True)
existing = list(user_dir.glob("*.jpg"))

print(f"✅ 이름: {teacher_name}")
print(f"📄 저장 경로: {user_dir}")
print(f"   기존 사진: {len(existing)}장")

In [ ]:
import time

existing  = list(user_dir.glob("*.jpg"))
next_idx  = len(existing) + 1
captured  = 0
saved_paths = []  # 삭제용 히스토리

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# 카메라 워밍업 (처음 몇 프레임은 검게 나올 수 있음)
print("카메라 초기화 중...")
for _ in range(20):
    cap.read()
print("준비 완료!")

DETECT_SKIP  = 5
frame_count  = 0
cached_faces = []

print(f"\n[{teacher_name}] 촬영 시작")
print("  SPACE: 카운트다운 후 촬영")
print("  d    : 마지막 사진 삭제")
print("  q    : 종료\n")

while True:
    ok, frame = cap.read()
    if not ok:
        continue
    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]
    frame_count += 1

    # InsightFace는 5프레임마다만 실행
    if frame_count % DETECT_SKIP == 0:
        cached_faces = app.get(frame)

    has_face = len(cached_faces) > 0
    color  = (0, 220, 80) if has_face else (60, 60, 220)
    status = f"face OK — SPACE to capture  ({len(existing) + captured}장)" \
             if has_face else "얼굴을 카메라에 맞춰 주세요"

    vis = frame.copy()
    for face in cached_faces:
        x1, y1, x2, y2 = map(int, face.bbox)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)

    cv2.putText(vis, f"Name: {teacher_name}", (12, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.putText(vis, status, (12, h - 16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    cv2.putText(vis, "SPACE:촬영  d:마지막삭제  q:종료", (12, h - 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)
    cv2.imshow("register", vis)

    key = cv2.waitKey(1) & 0xFF

    # ── SPACE: 3-2-1 카운트다운 후 촬영 ──────────────────────────────────
    if key == ord(" "):
        if not has_face:
            print("  ⚠️  얼굴 미감지 — 다시 시도하세요")
            continue

        cx, cy = w // 2, h // 2
        for cnt in [3, 2, 1]:
            ok2, cf = cap.read()
            cf = cv2.flip(cf, 1) if ok2 else frame.copy()
            for face in cached_faces:
                x1, y1, x2, y2 = map(int, face.bbox)
                cv2.rectangle(cf, (x1, y1), (x2, y2), (0, 220, 80), 2)
            cv2.putText(cf, str(cnt), (cx - 40, cy + 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 5.0, (0, 0, 0), 15)
            cv2.putText(cf, str(cnt), (cx - 40, cy + 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 5.0, (0, 220, 80), 8)
            cv2.imshow("register", cf)
            cv2.waitKey(1)
            time.sleep(1)

        ok2, clean_shot = cap.read()
        clean_shot = cv2.flip(clean_shot, 1) if ok2 else frame.copy()
        save_path = user_dir / f"{next_idx + captured:03d}.jpg"
        cv2.imwrite(str(save_path), clean_shot)
        saved_paths.append(save_path)
        captured += 1
        print(f"  ✅ [{captured}장] 저장: {save_path.name}")

        flash = clean_shot.copy()
        cv2.putText(flash, "OK!", (cx - 80, cy + 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 4.0, (0, 0, 0), 12)
        cv2.putText(flash, "OK!", (cx - 80, cy + 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 4.0, (0, 220, 80), 6)
        cv2.imshow("register", flash)
        cv2.waitKey(800)
        frame_count = 0

    # ── d: 마지막 사진 삭제 ──────────────────────────────────────────────
    elif key == ord("d"):
        if saved_paths:
            last = saved_paths.pop()
            if last.exists():
                last.unlink()
                captured -= 1
                print(f"  🗑️  삭제: {last.name}  (남은 촬영: {captured}장)")
            else:
                print("  ⚠️  파일이 이미 없습니다")
        else:
            print("  ⚠️  삭제할 사진이 없습니다")

    # ── q: 종료 ──────────────────────────────────────────────────────────
    elif key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

total = len(existing) + captured
print(f"\n[완료] '{teacher_name}' 총 {total}장 등록됨")

if CACHE_PATH.exists():
    CACHE_PATH.unlink()
    print("🔄 임베딩 캐시 초기화 — ai_node 실행 시 자동 재생성됩니다")

In [ ]:
# 등록 결과 확인
print("📋 현재 등록된 프로필:")
for person_dir in sorted(KNOWN_DIR.iterdir()):
    if person_dir.is_dir():
        photos = list(person_dir.glob("*.jpg")) + list(person_dir.glob("*.png"))
        print(f"  • {person_dir.name:20s} {len(photos)}장")

print()
print("▶ 실행 방법:")
print("  ROS_DOMAIN_ID=51 ros2 run pinky_yolo ai_node")